In [10]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

data_set = pd.read_excel(r"C:\Users\Gabriele\Desktop\Uni\big data\project 4\femicide_data.xlsx")

In [9]:

pio.renderers.default = "notebook_connected"

d = data_set.dropna(subset=["age", "type_of_femicide"]).copy()
d["tipo"] = d["type_of_femicide"].str.strip().str.lower()
d = d[d["tipo"].isin(["intimate", "non-intimate"])]
d["intimo"] = (d["tipo"] == "intimate").astype(int)

fasce = [0, 20, 25, 30, 40, 50, 120]
nomi = ["<20", "20-25", "25-30", "30-40", "40-50", "50+"]
d["fascia"] = pd.cut(d["age"], fasce, labels=nomi, right=False)

g = d.groupby("fascia", observed=True)["intimo"].agg(n="size", k="sum")
p = g["k"] / g["n"]

z = 1.96
centro = (p + z**2 / (2 * g["n"])) / (1 + z**2 / g["n"])
semi = z * np.sqrt(p * (1 - p) / g["n"] + z**2 / (4 * g["n"]**2)) / (1 + z**2 / g["n"])
lo, hi = np.maximum(centro - semi, 0), np.minimum(centro + semi, 1)

fig = go.Figure(go.Bar(
    x=g.index.astype(str), y=p,
    error_y=dict(type="data", symmetric=False, array=hi - p, arrayminus=p - lo),
    text=[f"{v:.0%}      <br>n={nn}        " for v, nn in zip(p, g["n"])], textposition="outside",
    marker_color="#1f3b73"))

fig.add_hline(y=d["intimo"].mean(), line_dash="dash", line_color="grey",
              annotation_text=f"media generale {d['intimo'].mean():.0%}")

fig.update_layout(
    title="Probabilità che l'autore sia un partner intimo,<br>per fascia d'età della vittima",
    xaxis_title="Età della vittima", yaxis_title="P(partner intimo)",
    yaxis=dict(tickformat=".0%", range=[0, 1.25]))
fig.show()


In [7]:

pio.renderers.default = "notebook_connected"

rng = np.random.default_rng(42)

tempi = np.sort(data_set["days_to_verdict"].dropna().values / 365.25)
n = len(tempi)

ecdf = lambda campione, x: np.searchsorted(np.sort(campione), x, side="right") / len(campione)

orizzonti = np.array([1, 2, 3, 5, 7, 10])
prob = ecdf(tempi, orizzonti)

B = 2000
boot = np.array([ecdf(rng.choice(tempi, size=n, replace=True), orizzonti) for _ in range(B)])
lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)

fig = go.Figure(go.Bar(
    x=[f"{o} anni" for o in orizzonti], y=prob,
    error_y=dict(type="data", symmetric=False, array=hi - prob, arrayminus=prob - lo),
    text=[f"{p:.0%}        " for p in prob], textposition="outside",
    marker_color="#c0392b",
    hovertemplate="entro %{x}: %{y:.0%}<extra></extra>"))

fig.update_layout(
    title=f"Probabilità che un caso arrivi a sentenza <br>entro un certo periodo",
    xaxis_title="Anni trascorsi dall'omicidio", yaxis_title="Probabilità",
    height=640,
    yaxis=dict(tickformat=".0%", range=[0, 1.15]))
fig.show()